In [ ]:
# STEP 1: Upload and load the dataset
import os, sys, json, zipfile, random
import numpy as np
import pandas as pd
import sklearn
from google.colab import files

SEED = 42
random.seed(SEED); np.random.seed(SEED)

CSV = "insider_threat_clean_dataset.csv"
if not os.path.exists(CSV):
    files.upload()                       # pick insider_threat_clean_dataset.csv

raw = pd.read_csv(CSV)
print("Rows, columns:", raw.shape)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

Saving insider_threat_clean_dataset.csv to insider_threat_clean_dataset.csv
Rows, columns: (118614, 22)
Python 3.13.16 | pandas 2.2.3 | scikit-learn 1.6.1


In [ ]:
# STEP 2: Data-quality audit (nothing is changed here, we only look)
print("Missing values :", int(raw.isna().sum().sum()))
print("Duplicate rows :", int(raw.duplicated().sum()))
print("\nTarget counts:")
print(raw["is_malicious"].value_counts())
print("Malicious %   :", round(raw["is_malicious"].mean() * 100, 2))

print("\nColumns that never change (useless):",
      [c for c in raw.columns if raw[c].nunique() == 1])

bad_campus = int((raw["num_unique_campus"] > raw["num_entries"]).sum())
print("Impossible rows (unique campuses > number of entries):", bad_campus)
print("Off-hours pages > total pages:", int((raw["num_printed_pages_off_hours"] > raw["total_printed_pages"]).sum()))
print("Trip day > 0 but not abroad  :", int(((raw["trip_day_number"] > 0) & (raw["is_abroad"] == 0)).sum()))

Missing values : 0
Duplicate rows : 0

Target counts:
is_malicious
0    112228
1      6386
Name: count, dtype: int64
Malicious %   : 5.38

Columns that never change (useless): ['late_exit_flag']
Impossible rows (unique campuses > number of entries): 974
Off-hours pages > total pages: 0
Trip day > 0 but not abroad  : 0


In [ ]:
# STEP 3: Build a proxy employee ID (the file has no employee column)
# Static attributes do not change from day to day, so rows sharing the same values
# very likely belong to the same employee.
STATIC_COLS = [
    "employee_department", "employee_campus", "employee_position",
    "employee_seniority_years", "is_contractor", "employee_classification",
    "has_foreign_citizenship", "has_criminal_record", "has_medical_history",
    "employee_origin_country",
]

df = raw.copy()
df["_gid"] = df.groupby(STATIC_COLS).ngroup()
df["employee_id"] = "EMP-" + (df["_gid"] + 1).astype(str).str.zfill(5)

rows_per_emp = df.groupby("employee_id").size()
insiders = df.loc[df["is_malicious"] == 1, "employee_id"].nunique()
print("Proxy employees          :", df["employee_id"].nunique())
print("Rows per employee (median):", int(rows_per_emp.median()), "| min:", int(rows_per_emp.min()), "| max:", int(rows_per_emp.max()))
print("Employees with malicious rows:", insiders)

Proxy employees          : 1654
Rows per employee (median): 66 | min: 26 | max: 163
Employees with malicious rows: 162


In [ ]:
# STEP 4: Remove sensitive columns, dead columns and identity-proxy columns
SENSITIVE = ["has_foreign_citizenship", "has_criminal_record", "has_medical_history", "employee_origin_country"]
DEAD      = ["late_exit_flag"]
PROFILE   = ["employee_department", "employee_campus", "employee_position"]
# Kept ONLY for an ablation experiment later (not used as model features by default):
ABLATION_STATIC = ["employee_seniority_years", "is_contractor", "employee_classification"]

df = df.drop(columns=SENSITIVE + DEAD + PROFILE + ["_gid"])

# Data-quality flag for the impossible rows (kept, not deleted)
df["dq_campus_gt_entries"] = (df["num_unique_campus"] > df["num_entries"]).astype(int)

print("Columns now:", list(df.columns))

Columns now: ['employee_seniority_years', 'is_contractor', 'employee_classification', 'total_printed_pages', 'num_printed_pages_off_hours', 'total_files_burned', 'burned_from_other', 'is_abroad', 'trip_day_number', 'hostility_country_level', 'num_entries', 'num_unique_campus', 'entry_during_weekend', 'is_malicious', 'employee_id', 'dq_campus_gt_entries']


In [ ]:
# STEP 5: Feature engineering (row-level ratios, so nothing is learned from other rows = no leakage)
df["off_hours_print_ratio"]  = df["num_printed_pages_off_hours"] / (df["total_printed_pages"] + 1)
df["files_burned_per_entry"] = df["total_files_burned"] / (df["num_entries"] + 1)

BEHAVIOUR_FEATURES = [
    "total_printed_pages", "num_printed_pages_off_hours", "off_hours_print_ratio",   # printing
    "total_files_burned", "files_burned_per_entry", "burned_from_other",              # data transfer
    "num_entries", "num_unique_campus", "entry_during_weekend",                       # physical access
    "is_abroad", "trip_day_number", "hostility_country_level",                        # travel
]
TARGET = "is_malicious"

print(df[BEHAVIOUR_FEATURES].describe().loc[["min", "mean", "max"]].round(2).T)

                             min   mean     max
total_printed_pages          0.0  13.71  728.00
num_printed_pages_off_hours  0.0   0.58  253.00
off_hours_print_ratio        0.0   0.02    0.69
total_files_burned           0.0   9.38  186.00
files_burned_per_entry       0.0   6.27  176.00
burned_from_other            0.0   0.01    1.00
num_entries                  0.0   0.75    4.00
num_unique_campus            0.0   0.60    3.00
entry_during_weekend         0.0   0.05    1.00
is_abroad                    0.0   0.04    1.00
trip_day_number              0.0   0.22   14.00
hostility_country_level      0.0   0.01    3.00


In [ ]:
# STEP 6: Split by EMPLOYEE (train / validation / test), stratified on the target
from sklearn.model_selection import StratifiedGroupKFold

X_idx = np.arange(len(df))
y = df[TARGET].values
g = df["employee_id"].values

# 4 folds -> 1 fold (25%) becomes TEST
outer = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(outer.split(X_idx, y, g))

# 5 folds on the remaining 75% -> 1 fold (15% of all) becomes VALIDATION
inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(inner.split(trainval_idx, y[trainval_idx], g[trainval_idx]))
train_idx, val_idx = trainval_idx[tr_rel], trainval_idx[va_rel]

train, val, test = df.iloc[train_idx].copy(), df.iloc[val_idx].copy(), df.iloc[test_idx].copy()

print(f"{'set':11s}{'rows':>9s}{'% rows':>9s}{'employees':>11s}{'insider emp.':>14s}{'malicious %':>13s}")
for name, d in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:11s}{len(d):9,d}{len(d)/len(df)*100:9.1f}{d['employee_id'].nunique():11,d}"
          f"{d.loc[d[TARGET]==1,'employee_id'].nunique():14,d}{d[TARGET].mean()*100:13.2f}")

set             rows   % rows  employees  insider emp.  malicious %
train         70,861     59.7        991            94         4.90
validation    17,978     15.2        247            23         5.82
test          29,775     25.1        416            45         6.26


In [ ]:
# STEP 7: Imbalance strategy = keep the real class ratio + class weights (no resampling)
n_neg = int((train[TARGET] == 0).sum()); n_pos = int((train[TARGET] == 1).sum())
scale_pos_weight = n_neg / n_pos
print("Train normal   :", n_neg)
print("Train malicious:", n_pos)
print("Imbalance ratio:", round(scale_pos_weight, 2), ": 1")
print("Validation and test keep the ORIGINAL distribution (never resampled).")

Train normal   : 67386
Train malicious: 3475
Imbalance ratio: 19.39 : 1
Validation and test keep the ORIGINAL distribution (never resampled).


In [ ]:
# STEP 8: Check that preprocessing is correct (the program stops with an error if something is wrong)
set_ids = {n: set(d["employee_id"]) for n, d in [("train", train), ("validation", val), ("test", test)]}
assert not (set_ids["train"] & set_ids["validation"]), "employees overlap: train/validation"
assert not (set_ids["train"] & set_ids["test"]),       "employees overlap: train/test"
assert not (set_ids["validation"] & set_ids["test"]),  "employees overlap: validation/test"
assert len(train) + len(val) + len(test) == len(df),   "rows lost or duplicated"

for n, d in [("train", train), ("validation", val), ("test", test)]:
    assert d[BEHAVIOUR_FEATURES].isna().sum().sum() == 0, n + " has missing values"
    assert np.isfinite(d[BEHAVIOUR_FEATURES].to_numpy(dtype=float)).all(), n + " has infinite values"
    assert d[TARGET].sum() > 0, n + " has no malicious rows"

leaked = set(SENSITIVE + DEAD + PROFILE) & set(df.columns)
assert not leaked, f"unwanted columns still present: {leaked}"
print("All checks passed: no employee overlap, no missing/infinite values, no sensitive columns.")

All checks passed: no employee overlap, no missing/infinite values, no sensitive columns.


In [ ]:
# STEP 9: Save the processed datasets, feature lists and a preprocessing report
KEEP = ["employee_id"] + BEHAVIOUR_FEATURES + ABLATION_STATIC + ["dq_campus_gt_entries", TARGET]

os.makedirs("processed", exist_ok=True)
train[KEEP].to_csv("processed/train.csv", index=False)
val[KEEP].to_csv("processed/validation.csv", index=False)
test[KEEP].to_csv("processed/test.csv", index=False)
# Isolation Forest learns from NORMAL rows only (no labels needed)
train.loc[train[TARGET] == 0, ["employee_id"] + BEHAVIOUR_FEATURES].to_csv("processed/if_train_normal_only.csv", index=False)

config = {
    "seed": SEED,
    "behaviour_features": BEHAVIOUR_FEATURES,
    "ablation_static_features": ABLATION_STATIC,
    "removed_sensitive": SENSITIVE, "removed_dead": DEAD, "removed_profile": PROFILE,
    "proxy_employee_key": STATIC_COLS,
    "split": "StratifiedGroupKFold by proxy employee: test 25%, validation ~15%, train ~60%",
    "imbalance_strategy": "original distribution + class weights",
    "train_scale_pos_weight": round(scale_pos_weight, 3),
    "rows": {"train": len(train), "validation": len(val), "test": len(test)},
    "library_versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "sklearn": sklearn.__version__},
}
with open("processed/preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

with zipfile.ZipFile("processed_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(os.listdir("processed")):
        z.write(os.path.join("processed", name), arcname=name)

print("Saved:", sorted(os.listdir("processed")))
files.download("processed_data.zip")

Saved: ['if_train_normal_only.csv', 'preprocessing_config.json', 'test.csv', 'train.csv', 'validation.csv']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# PART B - STEP 1: Load the processed data from Part A
import os, json, zipfile, random
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# If you are in a NEW Colab session, upload processed_data.zip when asked.
if not os.path.exists("processed/train.csv"):
    from google.colab import files
    files.upload()                                   # choose processed_data.zip
    with zipfile.ZipFile("processed_data.zip") as z:
        z.extractall("processed")

train = pd.read_csv("processed/train.csv")
val   = pd.read_csv("processed/validation.csv")
test  = pd.read_csv("processed/test.csv")
cfg   = json.load(open("processed/preprocessing_config.json"))

FEATURES = cfg["behaviour_features"]      # the 12 behaviour features
TARGET   = "is_malicious"
print("Train / validation / test rows:", len(train), len(val), len(test))
print("Candidate features (%d):" % len(FEATURES), FEATURES)
print("Feature selection uses ONLY the training set. Validation and test stay untouched.")

Train / validation / test rows: 70861 17978 29775
Candidate features (12): ['total_printed_pages', 'num_printed_pages_off_hours', 'off_hours_print_ratio', 'total_files_burned', 'files_burned_per_entry', 'burned_from_other', 'num_entries', 'num_unique_campus', 'entry_during_weekend', 'is_abroad', 'trip_day_number', 'hostility_country_level']
Feature selection uses ONLY the training set. Validation and test stay untouched.


In [11]:
# PART B - STEP 2: Look for features that are almost copies of each other (train only)
corr = train[FEATURES].corr().abs()
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i+1:] if corr.loc[a, b] >= 0.80]
print("Highly correlated pairs (|r| >= 0.80):")
for a, b, r in sorted(pairs, key=lambda t: -t[2]):
    print(f"  {a:30s} <-> {b:28s} r = {r}")
if not pairs:
    print("  none")

Highly correlated pairs (|r| >= 0.80):
  total_files_burned             <-> files_burned_per_entry       r = 0.9
  is_abroad                      <-> trip_day_number              r = 0.84
  num_entries                    <-> num_unique_campus            r = 0.83


In [12]:
# PART B - STEP 3: Grouped cross-validation setup + Gradient Boosting feature ranking
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score

y_tr = train[TARGET].values
g_tr = train["employee_id"].values
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(train, y_tr, g_tr))          # same folds reused everywhere (fair comparison)

def make_gbt():
    return HistGradientBoostingClassifier(class_weight="balanced", max_depth=6, learning_rate=0.1,
                                          max_iter=120, random_state=SEED)

imp = []
for k, (tr_i, va_i) in enumerate(folds, 1):
    m = make_gbt().fit(train.iloc[tr_i][FEATURES], y_tr[tr_i])
    r = permutation_importance(m, train.iloc[va_i][FEATURES], y_tr[va_i], scoring="average_precision",
                               n_repeats=5, random_state=SEED, n_jobs=-1)
    imp.append(r.importances_mean)
    print(f"fold {k}/5 done")

gbt_rank = pd.DataFrame({"feature": FEATURES, "importance_mean": np.mean(imp, axis=0),
                         "importance_std": np.std(imp, axis=0)}).sort_values("importance_mean", ascending=False)
print("\nGradient Boosting feature importance (drop in PR-AUC when the feature is shuffled):")
print(gbt_rank.round(4).to_string(index=False))

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done

Gradient Boosting feature importance (drop in PR-AUC when the feature is shuffled):
                    feature  importance_mean  importance_std
         total_files_burned           0.4334          0.1750
        total_printed_pages           0.1067          0.0628
      off_hours_print_ratio           0.0978          0.0703
          num_unique_campus           0.0396          0.0042
                num_entries           0.0251          0.0038
num_printed_pages_off_hours           0.0136          0.0161
     files_burned_per_entry           0.0107          0.0097
          burned_from_other           0.0099          0.0039
                  is_abroad           0.0053          0.0079
       entry_during_weekend           0.0051          0.0015
    hostility_country_level           0.0045          0.0075
            trip_day_number           0.0000          0.0009


In [13]:
# PART B - STEP 4: Pick the best 5 for Gradient Boosting and test how many features are really needed
CORR_LIMIT = 0.90                                   # skip a feature that is a near-copy of one already chosen
chosen = []
for f in gbt_rank["feature"]:
    if all(corr.loc[f, c] < CORR_LIMIT for c in chosen):
        chosen.append(f)
    if len(chosen) == 5:
        break
GBT_FEATURES = chosen

def cv_score(cols):
    oof = np.zeros(len(train))
    for tr_i, va_i in folds:
        m = make_gbt().fit(train.iloc[tr_i][cols], y_tr[tr_i])
        oof[va_i] = m.predict_proba(train.iloc[va_i][cols])[:, 1]
    thr = np.quantile(oof[y_tr == 0], 0.95)           # cutoff that gives a 5% false-positive rate
    return average_precision_score(y_tr, oof), roc_auc_score(y_tr, oof), (oof[y_tr == 1] >= thr).mean()

rows = []
for k in [3, 5, 8, len(FEATURES)]:
    cols = GBT_FEATURES if k == 5 else list(gbt_rank["feature"].iloc[:k])
    pr, roc, rec = cv_score(cols)
    rows.append({"features": k, "PR-AUC": pr, "ROC-AUC": roc, "recall@5%FPR": rec})
res = pd.DataFrame(rows)
print("Grouped 5-fold cross-validation on TRAIN (higher is better):")
print(res.round(3).to_string(index=False))
print("\nSelected Gradient Boosting features (5):", GBT_FEATURES)

Grouped 5-fold cross-validation on TRAIN (higher is better):
 features  PR-AUC  ROC-AUC  recall@5%FPR
        3   0.650    0.903         0.698
        5   0.693    0.920         0.745
        8   0.700    0.923         0.750
       12   0.714    0.930         0.771

Selected Gradient Boosting features (5): ['total_files_burned', 'total_printed_pages', 'off_hours_print_ratio', 'num_unique_campus', 'num_entries']


In [14]:
# PART B - STEP 5: Pick 5 features for Isolation Forest (forward selection, grouped CV, train only)
# Isolation Forest is trained on NORMAL rows only. Labels are used here ONLY to judge which
# features make the anomaly score separate malicious from normal days.
from sklearn.ensemble import IsolationForest

def make_if():
    return IsolationForest(n_estimators=100, random_state=SEED, n_jobs=-1)

def if_cv(cols):
    scores = np.zeros(len(train))
    for tr_i, va_i in folds:
        fit_rows = tr_i[y_tr[tr_i] == 0]                       # normal rows of the training fold only
        m = make_if().fit(train.iloc[fit_rows][cols])
        scores[va_i] = -m.score_samples(train.iloc[va_i][cols])  # higher = more unusual
    return average_precision_score(y_tr, scores), roc_auc_score(y_tr, scores)

IF_FEATURES = []
remaining = list(FEATURES)
for step in range(1, 6):
    best = None
    for f in remaining:
        pr, roc = if_cv(IF_FEATURES + [f])
        if best is None or pr > best[1]:
            best = (f, pr, roc)
    IF_FEATURES.append(best[0]); remaining.remove(best[0])
    print(f"step {step}: added {best[0]:30s} PR-AUC = {best[1]:.3f}  ROC-AUC = {best[2]:.3f}")

print("\nSelected Isolation Forest features (5):", IF_FEATURES)

step 1: added total_files_burned             PR-AUC = 0.497  ROC-AUC = 0.725
step 2: added is_abroad                      PR-AUC = 0.509  ROC-AUC = 0.752
step 3: added off_hours_print_ratio          PR-AUC = 0.530  ROC-AUC = 0.785
step 4: added entry_during_weekend           PR-AUC = 0.504  ROC-AUC = 0.803
step 5: added files_burned_per_entry         PR-AUC = 0.473  ROC-AUC = 0.794

Selected Isolation Forest features (5): ['total_files_burned', 'is_abroad', 'off_hours_print_ratio', 'entry_during_weekend', 'files_burned_per_entry']


In [15]:
# PART B - STEP 6: Sanity comparison for Isolation Forest (still grouped CV on train)
rows = []
for name, cols in [("selected 5", IF_FEATURES), ("all 12", FEATURES), ("the Gradient Boosting 5", GBT_FEATURES)]:
    pr, roc = if_cv(cols)
    rows.append({"feature set": name, "PR-AUC": pr, "ROC-AUC": roc})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nNo-skill PR-AUC (= malicious share in train):", round(y_tr.mean(), 3))

            feature set  PR-AUC  ROC-AUC
             selected 5   0.473    0.794
                 all 12   0.418    0.862
the Gradient Boosting 5   0.402    0.832

No-skill PR-AUC (= malicious share in train): 0.049


In [16]:
# PART B - STEP 7: Save the selected features
selection = {
    "gbt_features": GBT_FEATURES,
    "if_features": IF_FEATURES,
    "method": "grouped 5-fold CV on the training set only (StratifiedGroupKFold by proxy employee); "
              "GBT = permutation importance (PR-AUC) with correlation limit 0.90; IF = forward selection by PR-AUC of the anomaly score",
    "seed": SEED,
}
with open("processed/feature_selection.json", "w") as f:
    json.dump(selection, f, indent=2)
print(json.dumps(selection, indent=2))

from google.colab import files
files.download("processed/feature_selection.json")

{
  "gbt_features": [
    "total_files_burned",
    "total_printed_pages",
    "off_hours_print_ratio",
    "num_unique_campus",
    "num_entries"
  ],
  "if_features": [
    "total_files_burned",
    "is_abroad",
    "off_hours_print_ratio",
    "entry_during_weekend",
    "files_burned_per_entry"
  ],
  "method": "grouped 5-fold CV on the training set only (StratifiedGroupKFold by proxy employee); GBT = permutation importance (PR-AUC) with correlation limit 0.90; IF = forward selection by PR-AUC of the anomaly score",
  "seed": 42
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>